# Simple DP2 Injection Demo

Load one DP2 patch, then run three examples:

1. Inject 100 clusters in every band in `BANDS`.
2. Inject 100 clusters in `SELECTED_BAND`.
3. Run 10 rounds of 100 clusters in every band: 1,000 attempted injections per band.

Use your working CANFAR Rubin kernel and an up-to-date INJECT installation. Edit the data ID and bands below to match your data. Each round starts from the original image, not the previously injected image. Multiband runs use the same positions and apparent magnitudes, but a separate Rubin image and PSF in each band. This is a workflow demo, not a physical colour model or calibrated completeness test.

## 1. Imports
Run this in the Rubin kernel where INJECT is installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
from inject import ClusterConfig, InjectionConfig, InjectionPipeline, notebook_output_dir

## 2. Settings
Change these values. The data ID below comes from the earlier DP2 example; each chosen band must exist for that patch. DP2 calibrated nJy images use an AB zero point of about 31.4 and a pixel scale of about 0.2 arcsec/pixel. Verify these if using another product. Radii below are in pixels.

In [ ]:
BUTLER_REPO = 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp2/butler.yaml'
BUTLER_COLLECTIONS = 'dp2'
DATA_ID = {'skymap': 'lsst_cells_v2', 'tract': 7508, 'patch': 0}
BANDS = ['g', 'r', 'i']
SELECTED_BAND = 'r'
N_CLUSTERS = 100
N_ROUNDS = 10
OUTPUT_DIR = notebook_output_dir('simple_dp2_pipeline_demo')

config = InjectionConfig(
    bands=BANDS, band=SELECTED_BAND, n_clusters=N_CLUSTERS,
    cutout_size=800, edge_buffer=60, seed=42,
    use_actual_psf=True, add_noise=False, pixel_scale=0.2, zero_point=31.4,
    output_dir=str(OUTPUT_DIR / 'selected'),
    cluster_config=ClusterConfig(
        profile_type='king', mag_min=20.5, mag_max=24.5,
        r_half_min=2.0, r_half_max=6.0,
        concentration_min=5.0, concentration_max=20.0,
    ),
)

## 3. Load the data
Use the CANFAR session credentials. Load the bands once and reuse them below. The package handles DP2 CellCoadd conversion; no conversion code is needed here. Full coadds are fetched before cropping, so loading may take a while.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID, dataset_type='deep_coadd')

print({band: image.shape for band, image in pipe.images.items()})

## 4. Simple MCI detection
This detector uses matched filtering and MCI. Change its settings below or replace it with your own function later. `DETECTOR_PSF_FWHM` is a simple shared detection setting; injection still uses each band's actual local Rubin PSF.

Recovery uses the nearest detection within `MATCH_RADIUS` pixels. Matches are not one-to-one, and pre-existing sources are not removed, so treat the fractions as demo results rather than science completeness.

In [ ]:
import pandas as pd
from inject import ClusterRetrieval
from inject.detection import run_cluster_detection

MATCH_RADIUS = 5.0
DETECTOR_PSF_FWHM = 3.5

def detector(image):
    return run_cluster_detection(
        image, psf_fwhm=DETECTOR_PSF_FWHM, pixel_scale=config.pixel_scale,
        threshold_sigma=3.0, snr_min=3.0, mci_max=0.85,
        r_half_min=1.0, ellipticity_max=0.6, box_size=64,
        use_multiscale=True, use_mci=True, verbose=False,
    )

### Plot helper
`plot_run(...)` shows and saves the standard plot suite: original/injected/difference images, recovered (green) and missed (red) injections, 1D completeness versus magnitude and radius, 2D completeness, PSF-width diagnostics, and three postage stamps. Standalone original and injected PNGs are also saved. Empty completeness bins are unmeasured, not zero recovery. The small samples make these plots noisy.

In [ ]:
from dataclasses import replace
from inject.plotting import plot_completeness_1d, plot_completeness_2d

def plot_run(band, result, folder):
    folder = OUTPUT_DIR / folder
    view = InjectionPipeline(replace(config, band=band, bands=None, output_dir=str(folder)))
    view.load_data(image=pipe.images[band])
    view.injected_image = result['injected_image']
    view.injection_info = result['injection_info']
    view.retrieval = ClusterRetrieval.from_matched(view.injection_info)
    view.make_plots(output_dir=str(folder), show=True, save=True, n_stamps=3)
    vmin, vmax = np.nanpercentile(view.image, [1, 99])
    for name, image in [('original', view.image), ('injected', view.injected_image)]:
        plt.imsave(folder / f'{name}.png', image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    plt.close('all')

## 5. All-band injection, recovery and plots
The RGB overview shows original, injected, and recovered/missed positions. Its green/red markers use recovery in `SELECTED_BAND`, not a joint multiband detection rule. Red = i, green = r, blue = g; keep all three filters in `BANDS`.

Each filter also gets its full plot suite under `all_bands/<band>/plots`: 1D and 2D completeness, original/injected/difference, recovered/missed positions, PSF-width diagnostics, and postage stamps. Detection and completeness use the science arrays, never the RGB rendering. The display's median-sky subtraction and colour stretch do not change the science data.

In [ ]:
from astropy.visualization import make_lupton_rgb

RGB_BANDS = ('i', 'r', 'g')
if not set(RGB_BANDS).issubset(BANDS):
    raise ValueError('Keep g, r and i in BANDS for the colour image.')
if len({pipe.images[band].shape for band in RGB_BANDS}) != 1 or len({pipe.bboxes[band] for band in RGB_BANDS}) != 1:
    raise ValueError('The colour image needs matching band cutouts.')

all_bands = pipe.run_batch_multiband(
    n_iterations=1, n_per_iter=N_CLUSTERS, detector_fn=detector,
    checkpoint_dir=str(OUTPUT_DIR / 'all_bands'),
    store_images=True, use_psf_cache=False, n_workers=1, verbose=False,
)

all_recovery = []
for band in BANDS:
    result = all_bands[band][0]
    rows = result['injection_info']
    assert rows and all(row['psf_used'] == 'rubin' for row in rows), 'Check Rubin PSF evaluation.'
    retrieval = ClusterRetrieval(rows, result['detections'])
    retrieval.match_detections(match_radius=MATCH_RADIUS)
    all_recovery.append({'band': band, **retrieval.get_summary_statistics()})

channels = [np.nan_to_num(all_bands[band][0]['injected_image'] - np.nanmedian(pipe.images[band]),
                          nan=0.0, posinf=0.0, neginf=0.0) for band in RGB_BANDS]
original_channels = [np.nan_to_num(pipe.images[band] - np.nanmedian(pipe.images[band]),
                                   nan=0.0, posinf=0.0, neginf=0.0) for band in RGB_BANDS]
stretch = max(float(np.percentile(np.abs(channels), 99.5)), 1e-6)
rgb = make_lupton_rgb(*channels, stretch=stretch, Q=8)
original_rgb = make_lupton_rgb(*original_channels, stretch=stretch, Q=8)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for axis, image, title in zip(axes, [original_rgb, rgb, rgb],
                              ['Original RGB', 'Injected RGB', f'Recovered/missed in {SELECTED_BAND}']):
    axis.imshow(image, origin='lower')
    axis.set_title(title)
    axis.set_axis_off()
for detected, color, label in [(True, 'lime', 'Recovered'), (False, 'red', 'Missed')]:
    points = [row for row in all_bands[SELECTED_BAND][0]['injection_info'] if row['detected'] == detected]
    axes[2].scatter([row['x'] for row in points], [row['y'] for row in points],
                    facecolors='none', edgecolors=color, s=35, label=label)
axes[2].legend()
fig.savefig(OUTPUT_DIR / 'all_bands_rgb.png', dpi=150, bbox_inches='tight')
plt.show()
plt.imsave(OUTPUT_DIR / 'original_rgb.png', original_rgb, origin='lower')
plt.imsave(OUTPUT_DIR / 'injected_rgb.png', rgb, origin='lower')

all_recovery = pd.DataFrame(all_recovery)
all_recovery.to_csv(OUTPUT_DIR / 'all_bands_recovery.csv', index=False)
for band in BANDS:
    plot_run(band, all_bands[band][0], f'all_bands/{band}/plots')
all_recovery[['band', 'n_injected', 'n_detected', 'overall_completeness']]

## 6. Selected-band injection, recovery and plots
Use a clean original image for `SELECTED_BAND`. `detect_with()` runs MCI; `analyze()` matches detections to truth. Save the catalogs and the full plot suite under `selected/plots`, including 1D/2D completeness and green recovered/red missed positions on the injected image.

In [ ]:
pipe.image = pipe.images[SELECTED_BAND]
injected_image, injection_info = pipe.inject(band=SELECTED_BAND, verbose=False)
assert injection_info and all(row['psf_used'] == 'rubin' for row in injection_info), 'Check Rubin PSF evaluation.'
pipe.detect_with(detector)
selected_recovery = pipe.analyze(match_radius=MATCH_RADIUS)
pipe.save_results()

selected_result = {'injected_image': injected_image, 'injection_info': injection_info}
plot_run(SELECTED_BAND, selected_result, 'selected/plots')
pd.DataFrame([{'band': SELECTED_BAND, **selected_recovery}])[['band', 'n_injected', 'n_detected', 'overall_completeness']]

## 7. Batch recovery and plots: 10 rounds of 100 per band
Match separately within each band and round. The aggregate 1D and 2D completeness plots combine those existing flags across all 1,000 injections per band; no detections are cross-matched between rounds.

By default, the full image/position/postage-stamp suite shows round 1 of each band, while aggregate completeness uses all ten rounds. Files are under `batch/<band>/plots`; example images are in `round_01`, and pooled plots end in `_all_rounds.png`. Set `PLOT_ALL_ROUNDS=True` to save and display a full suite for every round. Do not interpret one round's image as containing all 1,000 sources.

Round images are temporarily retained for plotting, then released. Plotting every round costs extra time and disk space. The current multiband helper does not forward per-band masks, so this is not a mask-screened science run. All recovery fractions retain the nearest-neighbour/background-source limitations noted above.

In [ ]:
PLOT_ALL_ROUNDS = False
batch = pipe.run_batch_multiband(
    n_iterations=N_ROUNDS, n_per_iter=N_CLUSTERS, detector_fn=detector,
    checkpoint_dir=str(OUTPUT_DIR / 'batch'),
    store_images=True, use_psf_cache=False, n_workers=1, verbose=False,
)

batch_recovery = []
for band, rounds in batch.items():
    for result in rounds:
        rows = result['injection_info']
        assert rows and all(row['psf_used'] == 'rubin' for row in rows), 'Check Rubin PSF evaluation.'
        retrieval = ClusterRetrieval(rows, result['detections'])
        retrieval.match_detections(match_radius=MATCH_RADIUS)
        batch_recovery.append({
            'band': band, 'round': result['iteration'] + 1, **retrieval.get_summary_statistics(),
        })
    for result in (rounds if PLOT_ALL_ROUNDS else rounds[:1]):
        plot_run(band, result, f"batch/{band}/plots/round_{result['iteration'] + 1:02d}")
    matched_rows = [row for result in rounds for row in result['injection_info']]
    pooled = ClusterRetrieval.from_matched(matched_rows)
    folder = OUTPUT_DIR / 'batch' / band / 'plots'
    for name, plotter in [('completeness_1d', plot_completeness_1d), ('completeness_2d', plot_completeness_2d)]:
        fig, _ = plotter(pooled, config)
        fig.suptitle(f'{band} band: all {len(rounds)} rounds')
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        fig.savefig(folder / f'{name}_all_rounds.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
    for result in rounds:
        result.pop('injected_image', None)

batch_recovery = pd.DataFrame(batch_recovery)
batch_recovery.to_csv(OUTPUT_DIR / 'batch_recovery.csv', index=False)
batch_totals = batch_recovery.groupby('band')[['n_injected', 'n_detected']].sum()
batch_totals['overall_completeness'] = batch_totals['n_detected'] / batch_totals['n_injected']
print('Outputs:', OUTPUT_DIR)
batch_totals